# Working with the Overture extracts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kentstephen/bias-bounty-map-tutorial/blob/main/bias-bounty-explore-tutorial.ipynb)


Everything in this notebook runs against the live, public bucket. Nothing is downloaded to disk,
no credentials, no signup.

**Bias Bounties @ Scale** ships 13 layers for each of four US regions. Five are Overture,
the data being scored:

| layer | Overture theme / type | geometry |
|---|---|---|
| `overture-buildings` | `buildings` / `building` | polygons |
| `overture-roads` | `transportation` / `segment` (`subtype=road`) | lines |
| `overture-rail` | `transportation` / `segment` (`subtype=rail`) | lines |
| `overture-infrastructure` | `base` / `infrastructure` (stations, airports) | mixed |
| `overture-pois` | `places` / `place` | points |

The other eight are the reference layers you score Overture *against*:

| layer | source | geometry | what it is |
|---|---|---|---|
| `microsoft-buildings` | Microsoft GlobalML Building Footprints | polygons | reference baseline for the building coverage gap |
| `census-tiger-roads` | Census TIGER/Line | lines | reference baseline for the road gap |
| `census-acs-housing` | Census ACS 5-Year (B25001) | tract polygons | total housing units per tract |
| `census-cbp` | Census County Business Patterns | tract polygons | business establishment counts per tract |
| `hifld-hospitals` | USGS National Map structures | points | hospitals |
| `hifld-fire-stations` | USGS National Map structures | points | fire stations |
| `hifld-ems-stations` | USGS National Map structures | points | EMS / ambulance services |
| `hifld-schools` | USGS National Map structures | points | schools |

The path is always the same:

```
.../reference/<region>/<region>-<layer>.parquet
```

Three things this notebook covers, in order:

1. **Opening the files** (DuckDB and GeoPandas, one line each).
2. **The nested columns.** `names`, `categories` and `sources` are structs and lists, not strings.
   This is the first thing that bites people.
3. **An interactive map** that reads only the footprints in your current viewport, straight off
   the bucket.

## Setup

### Environment

Every dependency version is locked, in three places that agree:

- `pyproject.toml` + `uv.lock`: run `uv sync`, then open the notebook with `uv run jupyter lab`.
- The PEP 723 block in the next cell: run `uvx juv run bias-bounty-explore-tutorial.ipynb` and juv builds the exact environment on the fly.
- The `uv pip` line in the next cell: on Colab, run the cell once (and restart the runtime if Colab asks).


In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "arro3-core==0.8.1",
#     "duckdb==1.5.4",
#     "geoarrow-rust-io==0.6.1",
#     "geopandas==1.1.4",
#     "ipywidgets==8.1.8",
#     "lonboard==0.16.0",
#     "matplotlib==3.11.0",
#     "numpy==2.5.1",
#     "obstore==0.11.0",
#     "pandas==3.0.3",
#     "pyarrow==24.0.0",
# ]
# ///
# On Colab this installs the locked versions (uv, because pip takes minutes here).
# find_spec("google") first: find_spec("google.colab") raises off Colab, where no
# "google" package exists at all.
import importlib.util
if importlib.util.find_spec("google") and importlib.util.find_spec("google.colab"):
    %pip install -q uv
    !uv pip install --system -q arro3-core==0.8.1 duckdb==1.5.4 geoarrow-rust-io==0.6.1 geopandas==1.1.4 ipywidgets==8.1.8 lonboard==0.16.0 matplotlib==3.11.0 numpy==2.5.1 obstore==0.11.0 pandas==3.0.3 pyarrow==24.0.0
    # Colab only renders third-party widgets (lonboard is one) after this is enabled.
    from google.colab import output
    output.enable_custom_widget_manager()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 17.6 MB/s eta 0:00:00


In [2]:
import io, math, warnings
warnings.filterwarnings("ignore")
from concurrent.futures import ThreadPoolExecutor
from dataclasses import replace

from IPython.display import display

import duckdb
import geopandas as gpd
import ipywidgets as W
import numpy as np
import obstore
import pandas as pd
from arro3.core import Array, DataType, Table
from geoarrow.rust.io import GeoParquetDataset
from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer, SolidPolygonLayer, viz
from lonboard.basemap import CartoStyle, MaplibreBasemap
from lonboard.view_state import MapViewState
from matplotlib import colormaps
from matplotlib.colors import LogNorm
from obstore.store import S3Store
from pyarrow.fs import S3FileSystem

ORG, PRODUCT = "humane-intelligence", "bias-bounty-mapping-equity-challenge"
ROOT = f"{ORG}/{PRODUCT}"
REGIONS = ["maricopa-az", "northern-ca", "eastern-ok", "south-central-tx"]

# Two ways in to the same bytes. DuckDB reads over plain https; obstore/pyarrow use s3://.
HTTPS = f"https://data.source.coop/{ROOT}"

def url(region, layer, base=HTTPS, prefix="reference"):
    return f"{base}/{prefix}/{region}/{region}-{layer}.parquet"

# The bucket is public and needs no auth. But if you have AWS credentials configured (and
# plenty of people do, for work), pyarrow SIGNS the request, S3 rejects it, and you get a
# baffling ACCESS_DENIED / FileNotFoundError on a public file. An anonymous filesystem is
# immune either way. Note it takes the path WITHOUT the s3:// scheme.
ANON = S3FileSystem(anonymous=True, region="us-west-2")

def s3_path(region, layer, prefix="reference"):
    return f"us-west-2.opendata.source.coop/{ROOT}/{prefix}/{region}/{region}-{layer}.parquet"

# obstore + geoarrow-rust do the interactive map's viewport reads. Same bucket, same bytes.
store = S3Store("us-west-2.opendata.source.coop", region="us-west-2",
                endpoint="https://s3.us-west-2.amazonaws.com", skip_signature=True)

_objs, _ds = {}, {}

def objects(region, prefix="reference"):
    if (region, prefix) not in _objs:
        _objs[(region, prefix)] = {
            o["path"].split("/")[-1].removeprefix(f"{region}-").removesuffix(".parquet"): o
            for batch in obstore.list(store, f"{ROOT}/{prefix}/{region}/") for o in batch
            if o["path"].endswith(".parquet")     # strata/ ships a .csv twin of every table
        }
    return _objs[(region, prefix)]

def dataset(region, layer):
    if (region, layer) not in _ds:
        _ds[(region, layer)] = GeoParquetDataset.open([objects(region)[layer]], store=store)
    return _ds[(region, layer)]

aois = gpd.read_file(io.BytesIO(bytes(
    obstore.get(store, f"{ROOT}/boundaries/all-aois.geojson").bytes()
)))
print(len(objects("northern-ca")), "layers per region")

14 layers per region


### DuckDB

`httpfs` reads the files, `spatial` gives you the `ST_*` functions. Set the S3 region and
path-style addressing on every connection: the bucket name contains dots, so DuckDB's default
virtual-host addressing builds a hostname the TLS certificate doesn't cover.

In [ ]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("INSTALL spatial; LOAD spatial;")
con.sql("SET s3_region='us-west-2'; SET s3_url_style='path';")

con.sql(f"SELECT count(*) AS pois FROM '{url('northern-ca', 'overture-pois')}'").show()

### GeoPandas

Pass the `s3://` URI (not `https://`, which pyarrow has no driver for) and a `bbox` to read only
the row groups that intersect it. Here: 4.4M building footprints in the file, ~20k read.

In [ ]:
SAN_ANTONIO = (-98.52, 29.40, -98.46, 29.45)

gdf_buildings = gpd.read_parquet(
    s3_path("south-central-tx", "overture-buildings"), bbox=SAN_ANTONIO, filesystem=ANON,
)
# Or without S3FileSystem and the full path
# path = "s3://us-west-2.opendata.source.coop/humane-intelligence/bias-bounty-mapping-equity-challenge/reference/south-central-tx/south-central-tx-overture-buildings.parquet"
# gdf_buildings = gpd.read_parquet(path, bbox=SAN_ANTONIO)

print(f"{len(gdf_buildings):,} buildings, CRS {gdf_buildings.crs}")
gdf_buildings.plot(figsize=(6, 6), color="#E69F00", linewidth=0)


---
# 1. The nested columns

Overture does not flatten its attributes. `names` and `categories` are **structs**; `sources` is a
**list of structs**. Selecting them raw gives you nested objects, so the useful values need to be
reached into.

In [ ]:
con.sql(f'''
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT names, categories, sources FROM '{url("northern-ca", "overture-pois")}')
''').show(max_width=200)

### `names` and `categories` are structs: use dot access

`names.primary` is the one you want 95% of the time. `names.common` is a `MAP`, so it takes a key
(`['en']`), not an index. Only `names.rules` is a list.

In [ ]:
con.sql(f'''
    SELECT names.primary          AS name,
           names.common['en']     AS name_en,
           categories.primary     AS category,
           categories.alternate   AS also,
           confidence
    FROM '{url("northern-ca", "overture-pois")}'
    WHERE names.primary IS NOT NULL
    LIMIT 8
''').show(max_width=140)

### `sources` is a *list* of structs: index it or unnest it

`sources[1]` is the first source (DuckDB lists are 1-indexed). A feature can have several, so to
count them properly you `unnest`, which explodes one row per source.

In [ ]:
con.sql(f'''
    SELECT names.primary            AS name,
           len(sources)             AS n_sources,
           sources[1].dataset       AS first_dataset,
           sources[1].record_id     AS first_record_id
    FROM '{url("northern-ca", "overture-pois")}'
    WHERE names.primary IS NOT NULL
    LIMIT 8
''').show(max_width=140)

### The same structs in GeoPandas

You do not need DuckDB for this. `gpd.read_parquet` hands the structs back as plain Python
**dicts**, and the lists of structs as **arrays of dicts**. The `.str` accessor indexes both,
despite the name: it works on any object column, not just strings. Chain it to go a level deeper.

In [ ]:
CHICO = (-121.90, 39.68, -121.76, 39.80)

gdf_pois = gpd.read_parquet(
    s3_path("northern-ca", "overture-pois"), bbox=CHICO, filesystem=ANON,
    columns=["names", "categories", "sources", "confidence", "geometry"],
)

gdf_pois["name"]      = gdf_pois["names"].str["primary"]          # struct -> dict
gdf_pois["category"]  = gdf_pois["categories"].str["primary"]
gdf_pois["n_sources"] = gdf_pois["sources"].str.len()             # list of structs -> array of dicts
gdf_pois["first_ds"]  = gdf_pois["sources"].str[0].str["dataset"]  # chain to index into the first one

gdf_pois[["name", "category", "n_sources", "first_ds", "confidence"]].head(8)

`explode` is the pandas equivalent of DuckDB's `unnest`: one row per source. It works on a
GeoDataFrame and keeps the geometry, so you can go straight from this to a per-source map.

In [ ]:
gdf_pois.explode("sources")["sources"].str["dataset"].value_counts()

### Or get typed accessors, with the pyarrow backend

Dict indexing works but it is untyped, and it is slow on big frames. Ask pandas for Arrow-backed
columns instead and you get real `.struct` and `.list` accessors, which stay in Arrow the whole way.

The catch: `dtype_backend` is a **pandas** argument. `gpd.read_parquet` does not accept it, so this
idiom is for the non-spatial columns.

In [ ]:
df_pois = pd.read_parquet(
    s3_path("northern-ca", "overture-pois"), filesystem=ANON,
    columns=["names", "categories", "sources", "confidence"],
    dtype_backend="pyarrow",
)

print("fields on `names`:", df_pois["names"].struct.dtypes.index.tolist())

df_pois.assign(
    name      = df_pois["names"].struct.field("primary"),
    category  = df_pois["categories"].struct.field("primary"),
    n_sources = df_pois["sources"].list.len(),
    first_ds  = df_pois["sources"].list[0].struct.field("dataset"),
)[["name", "category", "n_sources", "first_ds", "confidence"]].head(8)

# The interactive map: draw a box, get the code

Pick a region, tick some layers, and they read straight off the bucket. The five big ones are
read for the viewport, so three of those at once is the limit; everything else is read whole and
is not limited. Plus one thing:
**draw a box on the map and it hands you the code to pull exactly those features.**

Hit the **▢ button at the lower-right of the map**, click two corners, and a box is drawn. The
panel below the map then fills with a **complete, copy-pasteable snippet**: the box and every
ticked layer are filled in, DuckDB reads them straight off the live bucket, and lonboard's `viz()`
puts the result back on a map. A **copy snippet** button puts it on the clipboard; **clear**
dismisses it.

`LAYERS` configures every layer in one place. Points, lines, and tracts are clickable for their
attributes; the Overture polygons draw as outlines, so to inspect buildings, draw a box and run
the snippet.

Two building layers ship for every region. Tick **overture-buildings** (amber) and
**microsoft-buildings** (blue) together, zoom past 13, then toggle or reorder them: where the two
sets of outlines do not agree is where the sources disagree about what is on the ground.

## Every layer, in one place

One entry per layer. `color` is a flat colour; `ramp` means it is a **choropleth**, coloured by that
column; `big` means it is read for the viewport rather than whole, which is what the three-layer
limit applies to. Everything else is a lonboard trait you can change without touching any other
code.

`BOUNDARIES` is a second, smaller table: administrative outlines from the bucket's `strata/`
prefix. They are read whole, like the eight un-flagged layers above, so they are not limited
either, and they are told apart by line weight rather than by colour. Everything shares one
draw-order list, so you decide what sits above what.

Colours are Okabe-Ito for the sparse layers, paled off for the five that cover the screen, and
they stay distinct under colour-vision deficiency. The choropleths use cividis and viridis, both
perceptually uniform and both readable for the same reason, one each so two can be on at once and
still be told apart.

In [ ]:
MAX_LAYERS = 3      # VIEWPORT-READ layers on at once. Each is its own read and its own GPU buffers.
MINZOOM = 13.0      # below this, a viewport-read layer shows its row-group boxes, not its features
BIG = 50_000        # rows: over this, a layer is read only for the viewport, never whole
HEAVY = 150_000     # rows on screen, all layers: past this the status line tells you to zoom in

# `big` marks a layer that is over BIG rows in every region, so it is read for the viewport and
# shows row-group boxes below MINZOOM. Those are the only ones MAX_LAYERS applies to: the rest are
# a few hundred to a few thousand features, read whole, and cost about what a boundary costs.
# The classification is not close in any region: the largest un-flagged layer is 17,387 rows
# (overture-rail, south-central-tx) and the smallest flagged one is 104,634 (overture-pois,
# northern-ca). Re-check it if the extracts are ever rebuilt.
# Saturation carries the read strategy, which is also the column split in the panel: the sparse
# whole-file layers are saturated, the viewport reads are pale, because those cover the screen and
# would otherwise shout over everything under them. Every value below is distinct; eleven flat
# colours is past what any palette keeps separable under colour-vision deficiency, so form does
# real work too (dots, lines, outlines) and nothing here is a red-versus-green pair.
LAYERS = {
    # name                    how it is drawn                                     what it is
    "census-acs-housing": {"ramp": "housing_units", "cmap": "cividis",
                           "opacity": 0.85, "line": [40, 40, 40]},
    "census-cbp":         {"ramp": "cbp_estab",     "cmap": "viridis",
                           "opacity": 0.85, "line": [40, 40, 40]},

    "hifld-hospitals":    {"color": "#D55E00", "opacity": 1.0,  "radius": 6},
    "hifld-fire-stations":{"color": "#E69F00", "opacity": 1.0,  "radius": 5},
    "hifld-ems-stations": {"color": "#CC79A7", "opacity": 1.0,  "radius": 5},
    "hifld-schools":      {"color": "#56B4E9", "opacity": 1.0,  "radius": 4},

    "overture-rail":      {"color": "#9E7BB5", "opacity": 0.9,  "width": 1.5},
    "overture-infrastructure": {"color": "#009E73", "opacity": 0.9, "radius": 4, "width": 1.5},

    "overture-buildings": {"color": "#F2C57C", "opacity": 0.4,  "big": True},
    "microsoft-buildings":{"color": "#8AB6D9", "opacity": 0.4,  "big": True},
    "overture-roads":     {"color": "#F5E77E", "opacity": 0.9,  "width": 1.5, "big": True},
    "census-tiger-roads": {"color": "#E7A9C6", "opacity": 0.9,  "width": 1.5, "big": True},
    "overture-pois":      {"color": "#A8E0CE", "opacity": 0.9,  "radius": 3, "big": True},
}

# Administrative boundaries, from the bucket's `strata/` prefix. These are outlines, not data:
# they are whole-file reads of a few hundred features, so they do NOT count against MAX_LAYERS, and
# like the AOI outline they stay out of the status counts and out of the drawn-box snippet. They do
# share the draw-order list with everything else, so you choose what sits above what.
#
# One ink, four weights. The nesting reads as line weight alone (the coarsest area heaviest, the
# tract mesh finest) so it survives both on the map and in an 11px legend chip, and nothing here
# depends on telling colours apart.
BOUNDARY_INK = "#D3DBE3"

BOUNDARIES = {
    # name                          line width   what it is
    "census-aiannh":              {"width": 2.5},   # AIAN/NH areas, the coarsest
    "census-tribal-subdivisions": {"width": 1.75},  # not published for every region
    "census-tribal-tracts":       {"width": 1.25},
    "census-tracts":              {"width": 0.75},  # the census tract mesh, the finest
}

# Two choropleths can be on at once, so they take different ramps: both are perceptually uniform
# and both survive a colour-vision-deficiency simulation, and they read as different at a glance.
MISSING = (120, 120, 120)      # a tract with no count is grey, NOT the dark end of the ramp

def ramp_fill(values, cmap="cividis"):
    '''count per tract -> (N, 3) uint8, log-normed. Log because tract counts are skewed: on a
    linear ramp almost every tract lands in the bottom bucket and the map goes flat.'''
    v = np.asarray(values, dtype="float64")
    finite = v[np.isfinite(v) & (v > 0)]
    norm = LogNorm(vmin=max(finite.min(), 1), vmax=max(finite.max(), 2))
    rgb = (colormaps[cmap](norm(np.where(v > 0, v, np.nan)))[:, :3] * 255).astype("uint8")
    rgb[~np.isfinite(v) | (v <= 0)] = MISSING
    return rgb

## Building a layer

Two paths, and the split is deliberate.

A **choropleth** is read whole through GeoPandas and built with `PolygonLayer.from_geopandas`.
These are 591 to 6,010 tracts: small enough to read in full.

Everything **else** is read arrow-native through geoarrow-rust, and only for the viewport when the
layer is big. That is the cloud-native read this whole project is about, and it is what lets you
put a million buildings on a map without downloading a million buildings.

In [ ]:
def choropleth(region, name, cfg):
    '''A census layer: one polygon per tract, filled by its count.'''
    gdf = gpd.read_parquet(s3_path(region, name), filesystem=ANON)
    return [PolygonLayer.from_geopandas(
        gdf,
        get_fill_color=ramp_fill(gdf[cfg["ramp"]], cfg.get("cmap", "cividis")),
        get_line_color=cfg.get("line", [40, 40, 40]),
        line_width_min_pixels=0.5,
        opacity=cfg["opacity"],
        pickable=True,
    )]

# Map each geoarrow geometry type to a layer kind, so we build the right layer class directly.
GEOM = {
    "geoarrow.point":           "point",
    "geoarrow.multipoint":      "point",
    "geoarrow.linestring":      "line",
    "geoarrow.multilinestring": "line",
    "geoarrow.polygon":         "area",
    "geoarrow.multipolygon":    "area",
}

def geom_kind(table):
    md = table.schema.field("geometry").metadata_str
    return GEOM.get(md.get("ARROW:extension:name"))

def one_layer(table, kind, cfg):
    """A table of one geometry type -> its lonboard layer. Every trait comes from cfg."""
    if kind == "point":
        return ScatterplotLayer(table=table, get_fill_color=cfg["color"],
                                radius_min_pixels=cfg.get("radius", 3),
                                opacity=cfg["opacity"], pickable=True)
    if kind == "line":
        return PathLayer(table=table, get_color=cfg["color"],
                         width_min_pixels=cfg.get("width", 1.5),
                         opacity=cfg["opacity"], pickable=True)
    # Overture polygons draw as outlines (filled=False). An outline repaints on a live camera move;
    # a solid fill does not in this lonboard version, so a fill vanishes until the next rebuild.
    return PolygonLayer(table=table, filled=False, stroked=True, pickable=True,
                        get_line_color=cfg["color"], line_width_min_pixels=2,
                        opacity=cfg["opacity"])

def features(region, name, cfg, bbox):
    """Everything that is not a choropleth. Read the viewport if the layer is big, else the whole file."""
    ds = dataset(region, name)
    table = ds.read(bbox=bbox) if (ds.num_rows > BIG and bbox) else ds.read()
    if table.num_rows == 0:
        return []

    kind = geom_kind(table)
    if kind:
        return [one_layer(table, kind, cfg)]

    # Mixed-geometry layer (overture-infrastructure): use viz() only to split it by geometry type,
    # then build each sub-layer here.
    return [one_layer(l.table, k, cfg)
            for l in viz(table).layers
            if (k := {"ScatterplotLayer": "point", "PathLayer": "line",
                      "PolygonLayer": "area", "SolidPolygonLayer": "area"}.get(type(l).__name__))]

def build(region, name, bbox=None):
    cfg = LAYERS[name]
    return choropleth(region, name, cfg) if "ramp" in cfg else features(region, name, cfg, bbox)

# layer_stack() runs on every camera move, so anything it returns has to be handed back, not
# rebuilt. Re-serialising hundreds of polygons per pan is what makes the map stall, and reusing the
# same instance across a `map.layers = [...]` swap is also what keeps deck from dropping its mesh.
_outlines = {}

def aoi_layer(region):
    if ("aoi", region) not in _outlines:
        _outlines[("aoi", region)] = PolygonLayer.from_geopandas(
            aois[aois.folder == region], filled=False, stroked=True,
            get_line_color=[255, 255, 255], line_width_min_pixels=1.5,
        )
    return _outlines[("aoi", region)]

def aoi_bbox(region):
    return tuple(aois[aois.folder == region].total_bounds)

# Context boundaries follow the AOI outline's pattern exactly: read whole, cached for the session,
# and rebuilt as a fresh layer on every draw. Rebuilding is safe here only because they are
# unfilled: the fill-drop this lonboard version has on a live layer swap cannot touch a stroke.
_bounds = {}

def boundary_gdf(region, name):
    if (region, name) not in _bounds:
        _bounds[(region, name)] = gpd.read_parquet(
            s3_path(region, name, prefix="strata"), filesystem=ANON)
    return _bounds[(region, name)]

def boundary_layer(region, name):
    if (name, region) not in _outlines:
        _outlines[(name, region)] = PolygonLayer.from_geopandas(
            boundary_gdf(region, name), filled=False, stroked=True,
            get_line_color=BOUNDARY_INK, line_width_min_pixels=BOUNDARIES[name]["width"],
            pickable=True,
        )
    return _outlines[(name, region)]

def has_boundary(region, name):
    """Not every region has every boundary: tribal subdivisions are published for two of the four."""
    return name in objects(region, prefix="strata")

# Below MINZOOM a big layer draws its row-group boxes (the parquet file's own spatial index)
# instead of its features: one footer read, and a viewport read fetches only the boxes it touches.
# Grey, so it reads as file structure rather than data.
BOX_LINE, BOX_FILL = "#B0B0B0", "#B0B0B012"

def partition_boxes(region, name):
    bounds = dataset(region, name).fragments[0].row_groups_bounds()
    n = len(bounds)
    table = Table.from_arrays(
        [bounds, Array([name] * n, DataType.string()), Array(np.arange(n), DataType.int32())],
        names=["geometry", "layer", "row_group"],
    )
    # not pickable: they cover the viewport and would block clicks on the features underneath
    return [PolygonLayer(table=table, get_fill_color=BOX_FILL, get_line_color=BOX_LINE,
                         stroked=True, filled=True, line_width_min_pixels=1, pickable=False)]

def is_big(name):
    """A viewport-read layer. Reading the flag beats opening thirteen parquet footers to ask."""
    return LAYERS[name].get("big", False)

## Finding a place

Photon (Komoot's geocoder, OpenStreetMap data, no API key). Type a place, press **Enter**, and
the hits land in a list, biased to the current region's bounding box. Pick one and the map goes
there.

In [ ]:
import json, urllib.parse, urllib.request

PHOTON = "https://photon.komoot.io/api"
JUMP_ZOOM = 14.0     # zoom to land on for a hit with no extent (an address, a shop)

def photon(query, bbox, limit=6):
    '''Geocode `query`, biased to `bbox`. Returns [(label, (lon, lat, extent)), ...] where
    extent is Photon's own bounding box for the feature (towns and counties have one; a single
    address does not), in its [minlon, maxlat, maxlon, minlat] order.'''
    params = urllib.parse.urlencode({
        "q": query,
        "limit": limit,
        "bbox": ",".join(f"{v:.4f}" for v in bbox),
    })
    with urllib.request.urlopen(f"{PHOTON}?{params}", timeout=6) as resp:
        features = json.load(resp)["features"]

    hits, seen = [], {}
    for f in features:
        p = f["properties"]
        # Photon labels vary by feature type: a town has name+state, an address has street+city.
        # dict.fromkeys de-dupes while keeping order ("Chico, Chico, CA" -> "Chico, CA").
        label = ", ".join(dict.fromkeys(
            x for x in (p.get("name"), p.get("city"), p.get("county"), p.get("state")) if x
        ))
        # ipywidgets.Select keys its options by label, so two identical labels collapse into one.
        seen[label] = seen.get(label, 0) + 1
        if seen[label] > 1:
            label = f"{label} ({seen[label]})"
        lon, lat = f["geometry"]["coordinates"]
        hits.append((label, (lon, lat, p.get("extent"))))
    return hits

def hit_to_view(hit):
    '''Where the camera should land for a geocoder hit.'''
    lon, lat, extent = hit
    if not extent:
        return MapViewState(longitude=lon, latitude=lat, zoom=JUMP_ZOOM)
    minlon, maxlat, maxlon, minlat = extent
    vs = bbox_to_view((minlon, minlat, maxlon, maxlat))
    # Never land below MINZOOM: you asked to go to a place, so you want to see its features, not
    # the row-group boxes. Never land above 16 either, or a small town fills the screen.
    return MapViewState(longitude=vs.longitude, latitude=vs.latitude,
                        zoom=min(max(vs.zoom, MINZOOM), 16.0))

## Viewport &harr; bounding box

Turning the camera into a box to read. This needs to know how wide the map is in pixels, which
lonboard does not report, so it is a constant. Too small and the edges of the map stay empty; too
large and you read features you do not draw. Erring large.

In [ ]:
VIEW_W, VIEW_H = 1400, 700

def _lat_to_y(lat):
    s = math.sin(math.radians(max(min(lat, 85.0), -85.0)))
    return 0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)

def _y_to_lat(y):
    return math.degrees(2 * math.atan(math.exp((0.5 - y) * 2 * math.pi)) - math.pi / 2)

def view_to_bbox(vs):
    world = 512 * (2 ** vs.zoom)
    half_lon = 360.0 * VIEW_W / world / 2
    yc, half_y = _lat_to_y(vs.latitude), VIEW_H / world / 2
    return (vs.longitude - half_lon, _y_to_lat(yc + half_y),
            vs.longitude + half_lon, _y_to_lat(yc - half_y))

def bbox_to_view(bbox, pad=1.15):
    xmin, ymin, xmax, ymax = bbox
    dlon = max((xmax - xmin) * pad, 1e-6)
    dy = max(abs(_lat_to_y(ymin) - _lat_to_y(ymax)) * pad, 1e-9)
    zoom = max(0.0, min(18.0, min(math.log2(360.0 * VIEW_W / 512 / dlon),
                                  math.log2(VIEW_H / 512 / dy))))
    return MapViewState(longitude=(xmin + xmax) / 2, latitude=(ymin + ymax) / 2, zoom=zoom)

def aoi_floor(region):
    '''Zoom that fits the whole AOI: the camera's min, so you can't zoom out past the loaded extent.'''
    return bbox_to_view(aoi_bbox(region)).zoom

## The map

Two update paths. A **layer-set change** (tick, untick, reorder, region, geocode) builds a new
`Map`; a **camera move** swaps the layers on the live map in place. Everything runs synchronously
in a browser-event handler (no threads, no timers), the one path Colab, JupyterLab, and VSCode
all deliver reliably.

In [ ]:
OPENING = ["census-acs-housing", "hifld-hospitals"]

# state["built"] maps layer name -> its current lonboard layers. state["bbox"] is the last box
# drawn on the map; read_key / read_bbox record what the last viewport read covered.
# state["layers"] is one ordered stack holding both kinds of thing: data layers (from LAYERS, read
# for the viewport, capped at MAX_LAYERS) and boundaries (from BOUNDARIES, read whole, uncapped).
# They share the order list so you can put either above the other; only the data layers are counted
# against the cap, counted in the status line, or written into the snippet.
state = {"region": "northern-ca", "layers": list(OPENING), "built": {}, "map": None,
         "view": None, "bbox": None,
         "read_key": None, "read_bbox": None,
         "counts": [], "note": "",
         "busy": False}      # True while a region is loading: see on_region

canvas = W.VBox([])       # holds the map
status = W.HTML()         # status line under the controls
snippet_out = W.VBox([])  # holds the drawn-box query, below the map

def set_status(view):
    """Write the status line from what is already built. Kept separate from the read because a
    camera move that needs no read still has to refresh the zoom readout: zooming IN always lands
    inside the box the last read covered, so without this the number never changes."""
    line = f"<b>zoom {view.zoom:.1f}</b>"
    if state["counts"]:
        line += " · " + " · ".join(state["counts"])
    if state["note"]:
        line += " · " + state["note"]
    status.value = line

def layer_stack():
    '''Everything to draw, bottom to top, in the order the draw-order list shows. A boundary builds
    from its cached gdf; a data layer comes from state["built"]. The AOI outline is pinned on top.'''
    return ([l for name in state["layers"]
             for l in ([boundary_layer(state["region"], name)] if name in BOUNDARIES
                       else state["built"].get(name, []))]
            + [aoi_layer(state["region"])])

def rebuild():
    """Build a new Map and swap it in, then CLOSE the one it replaced.

    Every Map is an anywidget model with its own WebAssembly instance and its own deck/luma
    context. Dropping the Python reference does not tear the frontend model down, so a Map built
    per interaction accumulates until the browser cannot allocate another WASM instance and the
    map dies with "Out of memory: Cannot allocate Wasm memory for new instance". Closing the old
    one explicitly is what frees it. Rebuild only when a Map genuinely has to be replaced;
    restack() is the cheap path for everything else."""
    old = state["map"]
    m = Map(layer_stack(), height=650, view_state=state["view"], show_tooltip=True,
            basemap=MaplibreBasemap(style=CartoStyle.DarkMatter))
    m.observe(on_camera, names="view_state")
    m.observe(on_select, names="selected_bounds")
    state["map"] = m
    canvas.children = (m,)
    if old is not None:
        old.close()          # after the swap, so the canvas is never empty

def restack():
    """Swap the layers on the existing map in place. Used for camera moves (pan/zoom)."""
    if state["map"] is None:
        rebuild()
    else:
        state["map"].layers = layer_stack()

# Unticking, reordering and toggling a boundary all leave every remaining layer as the instance it
# already was, so they need no new Map: swap the list on the live one. (A newly TICKED layer still
# goes through rebuild, because a freshly built filled layer is the case that does not paint until
# the camera moves. That is the lonboard bug the reload button exists for.)
repaint = restack

READ_PAD = 1.35     # read a little past the viewport, so a small pan needs no new read

def pad_bbox(b, f=READ_PAD):
    dx, dy = (b[2] - b[0]) * (f - 1) / 2, (b[3] - b[1]) * (f - 1) / 2
    return (b[0] - dx, b[1] - dy, b[2] + dx, b[3] + dy)

def covers(outer, inner):
    return (outer[0] <= inner[0] and outer[1] <= inner[1]
            and outer[2] >= inner[2] and outer[3] >= inner[3])

def render(view=None, fresh=True):
    """Read what the selection needs for this viewport and show it. fresh=True rebuilds the map
    (layer-set changes), fresh=False restacks the live one (camera moves)."""
    show = rebuild if fresh else restack
    try:
        region = state["region"]
        chosen = [n for n in state["layers"] if n in LAYERS]   # boundaries need no viewport read
        view = view or state["view"] or bbox_to_view(aoi_bbox(region))
        state["view"] = view = replace(view, min_zoom=aoi_floor(region))

        if not chosen:
            state["built"], state["read_key"], state["read_bbox"] = {}, None, None
            state["counts"], state["note"] = [], ""
            show()
            status.value = ("Pick a layer." if not state["layers"]
                            else "Boundaries only. Pick a layer to read features.")
            return

        # Below MINZOOM, big layers show their row-group boxes instead of every feature.
        boxed = [n for n in chosen if is_big(n) and view.zoom < MINZOOM]
        read = [n for n in chosen if n not in boxed]
        bbox = view_to_bbox(view)

        # Nothing to do if this viewport is still inside what the last read covered.
        if not fresh and state["read_key"] == (region, tuple(read), tuple(boxed)) \
                and state["read_bbox"] and covers(state["read_bbox"], bbox):
            set_status(view)      # nothing to read, but the zoom readout still moved
            return

        status.value = "<b>reading…</b> the read covers your whole viewport. Zoom in for a smaller one."
        # A camera move only changes what the big, viewport-read layers show. Reuse the cached
        # choropleth and small layers as-is, so their fills survive the live swap without a rebuild.
        # Reuse every built layer that is still valid, so ticking one layer reads that layer and
        # nothing else. A layer survives if it came from this region, was READ last time (a layer
        # that was boxed holds row-group boxes, not features, so crossing MINZOOM has to fetch it
        # for real), and either it is read whole or the last read still covers this viewport.
        # Reusing the instance is also what keeps a choropleth fill from dropping out on a live
        # swap.
        padded = pad_bbox(bbox)
        prev = state["built"]
        same_region = bool(state["read_key"]) and state["read_key"][0] == region
        was_read = set(state["read_key"][1]) if state["read_key"] else set()
        covered = bool(state["read_bbox"]) and covers(state["read_bbox"], bbox)
        reuse = [n for n in read if n in prev and n in was_read and same_region
                 and (covered or not is_big(n))]
        fetch = [n for n in read if n not in reuse]
        built = {n: prev[n] for n in reuse}
        with ThreadPoolExecutor(max_workers=max(len(fetch), 1)) as pool:
            built.update(zip(fetch, pool.map(lambda n: build(region, n, padded), fetch)))
        built.update({n: partition_boxes(region, n) for n in boxed})

        # What the built layers cover. If a viewport read was reused, only the part of the new box
        # its data actually reached is covered, so record the overlap rather than the new box.
        covers_now = padded
        if any(is_big(n) for n in reuse):
            old = state["read_bbox"]
            covers_now = (max(old[0], padded[0]), max(old[1], padded[1]),
                          min(old[2], padded[2]), min(old[3], padded[3]))
        state["built"] = built
        state["read_key"], state["read_bbox"] = (region, tuple(read), tuple(boxed)), covers_now
        show()

        # Status line: each layer with its row count, in draw order. Held in state so a camera
        # move that reads nothing can still redraw it with the new zoom.
        drawn = sum(l.table.num_rows for n in read for l in built[n])
        counts = [f"{sum(l.table.num_rows for l in built[n]):,} {n}" for n in read]
        counts += [f"{sum(l.table.num_rows for l in built[n])} row-group boxes ({n})" for n in boxed]

        if boxed:
            note = (f"<b>zoom past {MINZOOM:.0f}</b> to load "
                    f"{' and '.join(boxed)} for real.")
        elif drawn > HEAVY:
            note = (f"<b>{drawn:,} features on screen.</b> Zoom in, or untick a layer, "
                    f"for a faster map.")
        else:
            note = ""

        state["counts"], state["note"] = counts, note
        set_status(view)
    except Exception as exc:
        # Surface errors in the status line: a failure in a comm handler can otherwise go silent.
        status.value = f"<b style='color:#F0E442'>failed:</b> {type(exc).__name__}: {exc}"
        raise

def same_view(a, b):
    return a and b and (round(a.longitude, 6), round(a.latitude, 6), round(a.zoom, 4)) == \
                       (round(b.longitude, 6), round(b.latitude, 6), round(b.zoom, 4))

def on_camera(change):
    """Handle a camera move. The echo check ignores the event the map emits for a view we set."""
    if same_view(change["new"], state["view"]):
        return
    render(change["new"], fresh=False)


# ## Draw a box -> a copyable query
#
# The map's box button (lower-right) draws a bounding box. on_select turns it into a standalone
# snippet for the ticked layers, clipped to that box, and shows it below the map.

import html as _html

def copy_button(code):
    """Copy-to-clipboard button. The copy runs in an inline onclick, since Python cannot reach the
    browser clipboard from the kernel."""
    js = (f"const b = this; navigator.clipboard.writeText({json.dumps(code)})"
          ".then(() => { b.textContent = 'copied ✓';"
          " setTimeout(() => { b.textContent = 'copy snippet'; }, 1200); });")
    return W.HTML(f'<button class="bb-copy" onclick="{_html.escape(js, quote=True)}">copy snippet</button>')

# A real ipywidgets Button (unlike copy) because clearing has to reset state["bbox"] in the kernel.
clear_b = W.Button(description="clear", layout=W.Layout(width="70px"))

def on_clear(_):
    state["bbox"] = None
    if state["map"] is not None:
        state["map"].selected_bounds = None   # clear the map's own rectangle
    snippet_out.children = ()
    status.value = "snippet cleared. Draw another box (box button, lower-right) for a new one."

clear_b.on_click(on_clear)

def show_snippet():
    """Render the query for the last-drawn box below the map, kept in step with the ticked layers."""
    if not state["bbox"]:
        snippet_out.children = ()
        return
    picked = [n for n in state["layers"] if n in LAYERS]   # boundaries stay out of the snippet
    if not picked:
        snippet_out.children = (W.HTML("<i>Tick at least one layer, then draw a box.</i>"),)
        return
    code = render_snippet(state["region"], picked, state["bbox"])
    header = W.HBox(
        [W.HTML("<b>Query for the box you drew</b>"),
         copy_button(code), clear_b],
        layout=W.Layout(align_items="center"))
    header.add_class("bb-snippet")
    snippet_out.children = (W.HTML(PANEL_CSS),   # each Colab iframe needs its own copy of the CSS
                            header,
                            W.HTML(f'<pre class="bb-code">{_html.escape(code)}</pre>'))

def on_select(change):
    bounds = change["new"]
    if not bounds:                       # a fresh Map resets selected_bounds to None
        return
    state["bbox"] = tuple(bounds)
    show_snippet()
    status.value = ("<b>box drawn.</b> The snippet is below the map: copy it. "
                    "Draw again (box button, lower-right) to update it.")

## The query builder

In [ ]:
# Builds a standalone snippet: DuckDB reads the drawn box off the bucket (the bbox.* comparisons
# prune row groups), materializes it as a table named after the layer, and hands it to lonboard's
# viz(). The read is plain HTTPS: no region, no creds, no signing. One variable per ticked layer;
# point `url` at the one you want to map.

def layer_url(region, name):
    return f"https://data.source.coop/{ROOT}/reference/{region}/{region}-{name}.parquet"

def render_snippet(region, layers, bbox):
    xmin, ymin, xmax, ymax = bbox
    var = {n: f"{region}-{n}".replace("-", "_") for n in layers}
    width = max(len(v) for v in var.values())
    files = "\n".join(f'{var[n]:<{width}} = "{layer_url(region, n)}"' for n in layers)
    first = layers[0]
    count = {2: "two", 3: "three"}.get(len(layers), str(len(layers)))
    pick = (f"      # pick one of the {count} above and fill in here"
            if len(layers) > 1 else "")
    plural = "layers" if len(layers) > 1 else "layer"
    return f'''
import duckdb
from lonboard import viz

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs; INSTALL spatial; LOAD spatial")

# the box you drew (EPSG:4326)
xmin, ymin, xmax, ymax = {xmin:.6f}, {ymin:.6f}, {xmax:.6f}, {ymax:.6f}

# the {plural} you added (variable = filename)
{files}

url  = {var[first]}{pick}
name = url.split("/")[-1].removesuffix(".parquet")       # the filename, e.g. "{region}-{first}"

# CREATE ... AS materializes the clipped box into a table named after the layer, so con.table(name)
# hands it back to viz. The table name is quoted because the filename has dashes.
con.sql(f"""
    CREATE OR REPLACE TABLE '{{name}}' AS
    SELECT geometry::geometry AS geometry, * EXCLUDE (geometry)
    FROM read_parquet('{{url}}')
    WHERE bbox.xmin <= {{xmax}} AND bbox.xmax >= {{xmin}}
      AND bbox.ymin <= {{ymax}} AND bbox.ymax >= {{ymin}}
""")
con.sql(f"FROM '{{name}}'").show()
viz(con.table(name))'''

## Controls, and the map

If a tract fill drops out after a zoom or pan, click **⟳ reload layers** (lower right of the panel) to redraw the layers in place.

In [ ]:
# Styling for the control panel. The <style> ships as an HTML widget inside the panel (and the
# snippet gets its own copy) so it reaches the widgets even inside Colab's per-output iframes.
PANEL_CSS = '''
<style>
.bb-panel {
  background: #12151A; border: 1px solid #262C34; border-radius: 6px;
  padding: 14px 16px; margin-bottom: 8px; width: max-content;
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
}
.bb-panel .widget-label, .bb-panel label, .bb-panel select, .bb-panel button,
.bb-panel input[type="text"], .bb-panel .widget-html-content {
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace !important;
  font-size: 12px !important; color: #C6CDD6;
}
.bb-panel select, .bb-panel button, .bb-panel input[type="text"] {
  background: #1B2029; border: 1px solid #2E3743; border-radius: 4px; color: #C6CDD6;
}
.bb-panel input[type="text"]::placeholder { color: #5C6672; }
.bb-panel button:hover:enabled { background: #262E3A; }
.bb-panel button:disabled { opacity: 0.4; }
.bb-panel select option:checked { background: #2E3743; }
.bb-panel button.bb-reload { border-color: rgba(230, 159, 0, 0.38); }
.bb-head {
  font-size: 10px !important; letter-spacing: 0.12em; text-transform: uppercase;
  color: #6E7A88 !important; margin-bottom: 6px;
}
/* Fixed-height status strip, so changing feature counts do not reflow the panel. */
.bb-status {
  min-height: 20px; margin-top: 10px; padding-top: 8px; border-top: 1px solid #262C34;
  color: #8E99A6 !important;
}
.bb-status b { color: #E6EDF3; font-weight: 600; }
/* Swatch + checkbox as a centred flex row. */
.bb-swatch { display: flex; align-items: center; height: 100%; }
.bb-swatch .widget-html-content { display: flex; align-items: center; line-height: 1; }
.bb-row { align-items: center; }
.bb-row .widget-checkbox { margin: 0; }
/* Snippet header row (label + copy + clear). .bb-copy is a raw <button>; clear is a jupyter-button. */
.bb-snippet { margin: 8px 0 0 0; }
.bb-snippet .widget-html-content {
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px; color: #8E99A6;
}
.bb-snippet b { color: #E6EDF3; }
.bb-copy, .bb-snippet .jupyter-button {
  background: #1B2029; border: 1px solid #2E3743; border-radius: 4px; color: #C6CDD6;
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px;
  height: 24px; padding: 0 10px; margin-left: 10px; cursor: pointer;
}
.bb-copy:hover, .bb-snippet .jupyter-button:hover { background: #262E3A; }
/* The generated query <pre> below the map. */
.bb-code {
  background: #12151A; border: 1px solid #262C34; border-radius: 6px;
  padding: 12px 14px; margin: 8px 0 0 0; overflow-x: auto; max-width: 920px;
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px;
  color: #C6CDD6; line-height: 1.5;
}
</style>
'''

SWATCH_OFF = "#3A424D"

def ramp_chip(cmap):
    return "linear-gradient(90deg, {}, {}, {})".format(
        *["#%02X%02X%02X" % tuple(int(c) for c in np.array(colormaps[cmap](x)[:3]) * 255)
          for x in (0.0, 0.5, 1.0)])

def swatch(name, on):
    """Colour chip for a layer row: its flat colour, or its own ramp for a choropleth."""
    cfg = LAYERS[name]
    bg = SWATCH_OFF if not on else (ramp_chip(cfg["cmap"]) if "ramp" in cfg else cfg["color"])
    return f'<div style="width:11px;height:11px;border-radius:2px;background:{bg}"></div>'

# ---- layers ----------------------------------------------------------------------------------
checks, marks, rows = {}, {}, {}

def n_capped():
    """Only the viewport reads are capped. Everything read whole is free, boundaries included."""
    return sum(1 for n in state["layers"] if n in LAYERS and is_big(n))

def sync_cap():
    """At the cap, disable the unticked viewport reads. Ticked ones stay live, so you can trade
    one out. Nothing read whole is ever disabled."""
    at_cap = n_capped() >= MAX_LAYERS
    for name, box in checks.items():
        box.disabled = is_big(name) and at_cap and not box.value

def on_check(change):
    name = change["owner"].description
    if change["new"]:
        if is_big(name) and n_capped() >= MAX_LAYERS:
            # Mute the observer across the revert: setting .value here fires this handler again,
            # and the second pass would try to remove a name that was never added.
            box = change["owner"]
            box.unobserve(on_check, names="value")
            box.value = False
            box.observe(on_check, names="value")
            status.value = (f"<b>{MAX_LAYERS} viewport layers maximum.</b> "
                            f"Untick one to make room.")
            return
        state["layers"].append(name)      # newly ticked: drawn last, so it lands on top
    else:
        state["layers"].remove(name)
    marks[name].value = swatch(name, change["new"])
    sync_cap()
    sync_order(keep=name if change["new"] else None)
    if state["bbox"]:
        show_snippet()   # keep the query in step with what is ticked
    if change["new"]:
        render()          # a newly ticked layer has to be read before it can be drawn
    else:
        state["built"].pop(name, None)
        repaint()         # unticking is just a restack: drop it and repaint, no network

for name in LAYERS:
    on = name in state["layers"]
    checks[name] = W.Checkbox(value=on, description=name, indent=False,
                              layout=W.Layout(width="196px", margin="0", flex="0 0 auto"))
    marks[name] = W.HTML(swatch(name, on),
                         layout=W.Layout(width="19px", margin="0", flex="0 0 auto"))
    marks[name].add_class("bb-swatch")
    checks[name].observe(on_check, names="value")
    rows[name] = W.HBox([marks[name], checks[name]],
                        layout=W.Layout(align_items="center", margin="0", height="22px"))
    rows[name].add_class("bb-row")

# ---- context boundaries ----------------------------------------------------------------------
# Outside the layer cap, because these are outlines rather than data. Ticking one reads it (once
# per region, then cached) and repaints; it never triggers a viewport read.
ctx_checks, ctx_marks, ctx_rows = {}, {}, {}

# Chip heights step 4, 3, 2, 1 by rank rather than tracking the drawn width in pixels: the real
# widths (2.5 down to 0.75) collapse to two values once a browser rounds them, which made every
# boundary chip look the same. The chip shows the order, the map shows the weight.
CTX_RANK = {name: len(BOUNDARIES) - i for i, name in enumerate(BOUNDARIES)}

def ctx_swatch(name, on):
    """A rule rather than a block: one ink, stepped by where the boundary sits in the hierarchy."""
    bg = SWATCH_OFF if not on else BOUNDARY_INK
    return f'<div style="width:13px;height:{CTX_RANK[name]}px;background:{bg}"></div>'

def on_context(change):
    name = change["owner"].description
    if change["new"]:
        state["layers"].append(name)   # newly ticked: on top, then move it with the order list
    else:
        state["layers"].remove(name)
    ctx_marks[name].value = ctx_swatch(name, change["new"])
    try:
        if change["new"]:
            status.value = f"<b>reading…</b> {name}"
            boundary_gdf(state["region"], name)   # read here, so the repaint is instant
        sync_order(keep=name if change["new"] else None)
        repaint()                                 # reuses the built layers: no viewport read
        status.value = f"{name} {'on' if change['new'] else 'off'}."
    except Exception as exc:
        status.value = f"<b style='color:#F0E442'>failed:</b> {type(exc).__name__}: {exc}"
        raise

for name in BOUNDARIES:
    ctx_checks[name] = W.Checkbox(value=False, description=name, indent=False,
                                  layout=W.Layout(width="214px", margin="0", flex="0 0 auto"))
    ctx_marks[name] = W.HTML(ctx_swatch(name, False),
                             layout=W.Layout(width="19px", margin="0", flex="0 0 auto"))
    ctx_marks[name].add_class("bb-swatch")
    ctx_checks[name].observe(on_context, names="value")
    ctx_rows[name] = W.HBox([ctx_marks[name], ctx_checks[name]],
                            layout=W.Layout(align_items="center", margin="0", height="22px"))
    ctx_rows[name].add_class("bb-row")

def sync_context(region):
    """Boundaries are not published for every region (tribal subdivisions cover two of the four).
    Hide the rows this region has nothing for, rather than showing a box that cannot be ticked.
    Then pre-read what stays on, so switching region does not read inside the rebuild."""
    for name, box in ctx_checks.items():
        ok = has_boundary(region, name)
        if not ok and box.value:
            box.unobserve(on_context, names="value")
            box.value = False
            box.observe(on_context, names="value")
            ctx_marks[name].value = ctx_swatch(name, False)
            state["layers"].remove(name)
        ctx_rows[name].layout.display = "flex" if ok else "none"
    for name in state["layers"]:
        if name in BOUNDARIES:
            boundary_gdf(region, name)

# ---- region ----------------------------------------------------------------------------------
region_w = W.Dropdown(options=REGIONS, value=state["region"], description="",
                      layout=W.Layout(width="215px"))

def on_region(change):
    """Load a region, with the search locked until it lands.

    A region change and a geocode jump are two camera destinations, and each one builds a whole
    new Map. Firing them together churns WebAssembly instances faster than the browser reclaims
    them, which is what takes the map out. Two halves to the lock: `busy` is the one that actually
    holds, because a keystroke already on its way still gets delivered and handled; disabling the
    widgets is the half you can see."""
    state["region"] = change["new"]
    state["busy"] = True
    search_w.disabled = hits_w.disabled = True
    try:
        sync_context(change["new"])       # a boundary this region lacks switches itself off
        sync_order()                      # which may have dropped it out of the draw order
        search_w.value = ""               # the old region's hits are meaningless here
        state["bbox"] = None              # a box drawn in the old region is meaningless here
        show_snippet()
        show_hits(())
        render(bbox_to_view(aoi_bbox(change["new"])))
    finally:                              # a failed read must not leave the search locked out
        state["busy"] = False
        search_w.disabled = hits_w.disabled = False

region_w.observe(on_region, names="value")

# ---- find a place ----------------------------------------------------------------------------
search_w = W.Text(placeholder="town, county, address… then Enter", continuous_update=False,
                  description="", layout=W.Layout(width="215px"))
hits_w = W.Select(options=(), rows=6, description="",
                  layout=W.Layout(width="215px", display="none", margin="6px 0 0 0"))

def show_hits(hits):
    """Fill the results list, or hide it when empty. Mute the value observer across the swap so
    setting .options does not fire a camera jump."""
    hits_w.unobserve(on_hit, names="value")
    hits_w.options = tuple(hits)
    hits_w.value = None
    hits_w.observe(on_hit, names="value")
    hits_w.layout.display = "flex" if hits else "none"

def on_search(change):
    """Runs once per search: the box syncs its value on Enter (continuous_update=False)."""
    if state["busy"]:                     # queued from before the region change, no longer meant
        return
    query = change["new"].strip()
    if len(query) < 3:                    # 1-2 letters match half the country
        show_hits(())
        return
    try:
        hits = photon(query, aoi_bbox(state["region"]))
    except Exception as exc:              # a geocoder outage must not take the map down
        show_hits(())
        status.value = f"geocoder unavailable ({type(exc).__name__}). The map still works."
        return
    show_hits(hits)
    if not hits:
        status.value = f"no match for <b>{query}</b> in {state['region']}."

def on_hit(change):
    if change["new"] is None or state["busy"]:
        return
    render(hit_to_view(change["new"]))    # jump straight to the place

search_w.observe(on_search, names="value")
hits_w.observe(on_hit, names="value")

# ---- draw order ------------------------------------------------------------------------------
# state["layers"] is draw order (first = underneath). The widget shows the reverse, so the top of
# the list is the top layer. Every conversion between the two happens here.
order_w = W.Select(options=tuple(reversed(state["layers"])), rows=7, description="",
                   layout=W.Layout(width="215px"))
fwd_b = W.Button(description="▲ bring forward", layout=W.Layout(width="105px", margin="6px 4px 0 0"))
back_b = W.Button(description="▼ send back", layout=W.Layout(width="105px", margin="6px 0 0 0"))
reload_b = W.Button(description="⟳ reload layers",
                    tooltip="Redraw the layers in place. Brings back a choropleth fill that dropped out on a zoom/pan.",
                    layout=W.Layout(width="214px", margin="28px 0 0 0"))
reload_b.add_class("bb-reload")

def sync_order(keep=None):
    """Rebuild the order list from state["layers"], keeping the current selection. Mute the observer
    across the swap so setting .options does not move the highlight."""
    keep = keep if keep is not None else order_w.value
    order_w.unobserve(sync_buttons, names="value")
    order_w.options = tuple(reversed(state["layers"]))
    order_w.value = keep if keep in state["layers"] else None
    order_w.observe(sync_buttons, names="value")
    sync_buttons()

def sync_buttons(*_):
    sel, layers = order_w.value, state["layers"]
    fwd_b.disabled = sel is None or layers.index(sel) == len(layers) - 1
    back_b.disabled = sel is None or layers.index(sel) == 0

def move(step):
    """Move the selected layer in draw order (+1 = toward the top, -1 = toward the bottom). No
    network: the layers are already built, so this just reshuffles and repaints."""
    def handler(_):
        layers, sel = state["layers"], order_w.value
        if sel is None:
            return
        i, j = layers.index(sel), layers.index(sel) + step
        if not 0 <= j < len(layers):
            return
        layers[i], layers[j] = layers[j], layers[i]
        sync_order(keep=sel)      # keep it selected: you usually click again
        repaint()
    return handler

order_w.observe(sync_buttons, names="value")
fwd_b.on_click(move(+1))          # forward on the map = later in the draw order
back_b.on_click(move(-1))

def on_reload(_):
    """Redraw in place so a dropped fill repaints. Reuses the built layers, no read."""
    rebuild()
    status.value = "layers reloaded."

reload_b.on_click(on_reload)

# ---- panel -----------------------------------------------------------------------------------
def column(title, *widgets):
    return W.VBox([W.HTML(f'<div class="bb-head">{title}</div>'), *widgets],
                  layout=W.Layout(margin="0 28px 0 0"))

panel = W.VBox([
    W.HTML(PANEL_CSS),                    # the CSS lives inside the panel's own (i)frame
    W.HBox([
        W.VBox([
            column("Set region", region_w),
            column("Find a place", search_w, hits_w),
        ], layout=W.Layout(margin="0 28px 0 0")),
        # Split by what a layer costs, not by which prefix it came from. Left: the five that are
        # read for the viewport and box up below MINZOOM. Right: everything read whole, which is
        # eight of the reference layers plus the boundaries.
        column("Viewport reads", W.HTML(f'<div class="bb-note">Limit {MAX_LAYERS}</div>'),
               *[rows[n] for n in LAYERS if is_big(n)]),
        column("Read whole", W.HTML('<div class="bb-note">No limit</div>'),
               *[rows[n] for n in LAYERS if not is_big(n)], *ctx_rows.values()),
        column("Draw order (top = on top)", order_w, W.HBox([fwd_b, back_b]), reload_b),
    ]),
    status,
])
panel.add_class("bb-panel")
status.add_class("bb-status")

sync_cap()
sync_context(state["region"])
sync_order()
render()    # build the opening map; the next cell displays it

In [ ]:
# Display-only cell (kept separate so Colab renders it reliably).
#
# Draw a box on the map with the ▢ button (lower-right): click two corners and a copy-pasteable
# query for the ticked layers, clipped to that box, appears below the map.
display(panel, canvas, snippet_out)

## What the box hands you

This is the snippet the panel produces, verbatim, for a box over downtown Oklahoma City
with **overture-buildings** ticked in `eastern-ok` (8,834 footprints). It is commented out so the
notebook stays runnable end to end; uncomment it, or paste it into a fresh cell or a fresh
notebook. The only dependencies are `duckdb` and `lonboard`.

The last line is lonboard's [`viz()`](https://developmentseed.org/lonboard/latest/api/viz/),
which takes the DuckDB relation directly and picks a layer type from the geometry. Its
[DuckDB example](https://developmentseed.org/lonboard/latest/examples/duckdb/) covers the same
pattern, and the rest of the [lonboard examples](https://developmentseed.org/lonboard/latest/examples/)
show how to style the result once you have it.

A useful tool is a bounding box generator [like this one](https://boundingbox.klokantech.com/). Find your box and select CSV and paste the value into `xmin, ymin, xmax, ymax =`.


In [ ]:
# The snippet the box writes, for downtown Oklahoma City + overture-buildings in eastern-ok.
# Uncomment to run (every line, including the triple-quoted SQL).
# import duckdb
# from lonboard import viz
#
# con = duckdb.connect()
# con.sql("INSTALL httpfs; LOAD httpfs; INSTALL spatial; LOAD spatial")
#
# # the box you drew (EPSG:4326)
# xmin, ymin, xmax, ymax = -97.540000, 35.450000, -97.490000, 35.490000
#
# # the layer you added (variable = filename)
# eastern_ok_overture_buildings = "https://data.source.coop/humane-intelligence/bias-bounty-mapping-equity-challenge/reference/eastern-ok/eastern-ok-overture-buildings.parquet"
#
# url  = eastern_ok_overture_buildings
# name = url.split("/")[-1].removesuffix(".parquet")       # the filename, e.g. "eastern-ok-overture-buildings"
#
# # CREATE ... AS materializes the clipped box into a table named after the layer, so con.table(name)
# # hands it back to viz. The table name is quoted because the filename has dashes.
# con.sql(f"""
#     CREATE OR REPLACE TABLE '{name}' AS
#     SELECT geometry::geometry AS geometry, * EXCLUDE (geometry)
#     FROM read_parquet('{url}')
#     WHERE bbox.xmin <= {xmax} AND bbox.xmax >= {xmin}
#       AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
# """)
# con.sql(f"FROM '{name}'").show()
# viz(con.table(name))


In [3]:
print(list(objects("northern-ca").keys()))

['census-acs-housing', 'census-cbp', 'census-tiger-roads', 'hifld-ems-stations', 'hifld-fire-stations', 'hifld-hospitals', 'hifld-schools', 'microsoft-buildings', 'overture-buildings', 'overture-infrastructure', 'overture-pois', 'overture-rail', 'overture-roads-unfiltered', 'overture-roads']


In [4]:
for batch in obstore.list(store, f"{ROOT}/reference/northern-ca/"):
    for o in batch:
        print(o["path"])

humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern-ca-census-acs-housing.csv
humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern-ca-census-acs-housing.parquet
humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern-ca-census-cbp.csv
humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern-ca-census-cbp.parquet
humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern-ca-census-tiger-roads.parquet
humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern-ca-hifld-ems-stations.csv
humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern-ca-hifld-ems-stations.parquet
humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern-ca-hifld-fire-stations.csv
humane-intelligence/bias-bounty-mapping-equity-challenge/reference/northern-ca/northern

In [5]:
import io
data = obstore.get(store, f"{ROOT}/reference/northern-ca/northern-ca-sample-submission.csv").bytes()
sub = pd.read_csv(io.BytesIO(bytes(data)), dtype={"GEOID": str})
print(sub.shape)
print(sub.columns.tolist())
sub.head()

(591, 5)
['GEOID', 'transport_gap', 'building_gap', 'poi_gap', 'coverage_gap_score']


,GEOID,transport_gap,building_gap,poi_gap,coverage_gap_score
0,06007000102,0.0,0.0,0.0,0.0
1,06007000103,0.0,0.0,0.0,0.0
2,06007000104,0.0,0.0,0.0,0.0
3,06007000201,0.0,0.0,0.0,0.0
4,06007000202,0.0,0.0,0.0,0.0


In [7]:
print(sub[["transport_gap", "building_gap", "poi_gap", "coverage_gap_score"]].describe())
print((sub[["transport_gap", "building_gap", "poi_gap", "coverage_gap_score"]] == 0).all())

       transport_gap  building_gap  poi_gap  coverage_gap_score
count          591.0         591.0    591.0               591.0
mean             0.0           0.0      0.0                 0.0
std              0.0           0.0      0.0                 0.0
min              0.0           0.0      0.0                 0.0
25%              0.0           0.0      0.0                 0.0
50%              0.0           0.0      0.0                 0.0
75%              0.0           0.0      0.0                 0.0
max              0.0           0.0      0.0                 0.0
transport_gap         True
building_gap          True
poi_gap               True
coverage_gap_score    True
dtype: bool
